# Exploring different features and model to select the best

Generating the report for my data

In [17]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

In [3]:
titanic_df = pd.read_csv("F://Titanic_Project//Data//train (2).csv")
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
import sweetviz as sv

report = sv.analyze(titanic_df)
report.show_html("F://Titanic_Project//Reports//raw_report.html" , open_browser=False)

Using the report and studying the data following things are identified

- Age is skewed right have to use np.log function to fix it
- Age has 20% missing values have to fix it
- Age has fractions for childerens under 1 
- sibling spouse and parent are in the sepearate coloumn can join them to make the family coloumn
- fare is also highly skewed have to fix it
- Cabin has large missing value so can create a falg for cabin present or not present
- embarked has two missing values can be filled with the mode

In [22]:
x_temp = titanic_df.copy()
x_temp["Family"] = x_temp["SibSp"] + x_temp["Parch"] + 1
x_temp["Is_married"] = x_temp["Name"].str.contains("Mrs", na=False).astype(int)
x_temp["Family_size"] = np.where(
    x_temp["Family"] >= 4,
    "Large",
    "Small"
)
x_temp["Is_alone"] = np.where(
    x_temp["Family"] == 1,
    1,
    0
)
x_temp["Age_bracket"] = pd.cut(x_temp['Age'], bins=[-np.inf, 12, 18, 60, np.inf], labels=['child','teen','adult','senior'])
x_temp["Has_cabin"] = x_temp["Cabin"].notna().astype(int)
x_temp["Fare_bracket"] = pd.qcut(x_temp["Fare"] ,q=4 , labels=["Low_fare" , "Medium_fare" , "High_fare" , "Very_high_fare"], duplicates='drop')
x_temp.drop(columns=["PassengerId" , "Name" , "Age" , "SibSp" , "Parch" , "Ticket" , "Fare" , "Cabin" , "Family"],inplace=True)
x_temp

,Survived,Pclass,Sex,Embarked,Is_married,Family_size,Is_alone,Age_bracket,Has_cabin,Fare_bracket
0,0,3,male,S,0,Small,0,adult,0,Low_fare
1,1,1,female,C,1,Small,0,adult,1,Very_high_fare
2,1,3,female,S,0,Small,1,adult,0,Medium_fare
3,1,1,female,S,1,Small,0,adult,1,Very_high_fare
4,0,3,male,S,0,Small,1,adult,0,Medium_fare
...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,S,0,Small,1,adult,0,Medium_fare
887,1,1,female,S,0,Small,1,adult,1,High_fare
888,0,3,female,S,0,Large,0,NaN,0,High_fare
889,1,1,male,C,0,Small,1,adult,1,High_fare


Following is the feature engineering that i have done based on the coloumns now i will add all this to the function that i can apply to the data 
- Since Embarked and Age need to fill the missing values before applying the function i will make a seprate pipeline to fill these values first

In [ ]:
def transform_data(df):
    X = df.copy()
    X["Family"] = X["SibSp"] + X["Parch"] + 1
    X["Is_married"] = X["Name"].str.contains("Mrs", na=False).astype(int)
    X["Family_size"] = np.where(
        X["Family"] >= 4,
        "Large",
        "Small"
    )
    X["Is_alone"] = np.where(
        X["Family"] == 1,
        1,
        0
    )
    X["Age_bracket"] = pd.cut(X['Age'], bins=[-np.inf, 12, 18, 60, np.inf], labels=['child','teen','adult','senior'])
    X["Has_cabin"] = X["Cabin"].notna().astype(int)
    X["Fare_bracket"] = pd.qcut(X["Fare"] ,q=4 , labels=["Low_fare" , "Medium_fare" , "High_fare" , "Very_high_fare"], duplicates='drop')
    X.drop(columns=["PassengerId" , "Name" , "Age" , "SibSp" , "Parch" , "Ticket" , "Fare" , "Cabin" , "Family"],inplace=True)    
    return X